# RNN
- RNN(yinelemeli sinir ağı)  ,sıralı veriler üzerinde çalışan özel bir derşn öğrenme modelidir.
- önceki girdileri hatırlayarak ve bu bilgileri kullanrrak sonraki çıktıları tahmin edder
- bu sayede doğal dil işleme (metin çeviri, duygu analizi), konuşma tanıma ve zaman serisi analizi gibi alanlarda yaygın olarak kullanılır
- sıralı veriye okdaklanma: rnn ler metinler , esler gibi sırasal yapıa sahip verilerle çalışmya özel olarak tasarlanmıştır
- hafıza mekanızması: bu ağlar, önceki bilgileri bir tğr hafıza hücresinde saklar ve bu sayede uzun süreli bağımlılıkalr yakalayabilir
- geniş uygulama alanları: doğal dil işleme, makine çevirsi,duygu analizi,konuşma tanıma ve hatta müzik besteleme gibi birçok alanda kullanılır
- zaman serisi, metin serisi, ses gibi sıralı verilerle kullanılır
- aynı ağırlıkarla zaman içinde tekrar eden yaoılaar(ağ belleği gibi)
- matematiksel olarak: ht = tanh(W * xt + U * ht-1 + b)
- özetle rnn ler sıralı veriler üzerindeki karmaşık yapıları anlamak ve tahmin etmek için kullanılan güçlü bir araçtır
-

In [ ]:
#Temel RNN
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Veri setini yükle
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
data = pd.read_csv(url, usecols=[1])
data = data.values.astype('float32')

In [ ]:
# Normalizasyon
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(data)

In [ ]:
# Eğitim/test ayrımı
train_size = int(len(dataset) * 0.8)
train, test = dataset[:train_size], dataset[train_size:]

# Zaman penceresi oluşturma
def create_dataset(dataset, time_step=1):
    dataX, dataY = [], []
    for i in range(len(dataset) - time_step - 1):
        dataX.append(dataset[i:(i + time_step), 0])
        dataY.append(dataset[i + time_step, 0])
    return np.array(dataX), np.array(dataY)

time_step = 10
X_train, y_train = create_dataset(train, time_step)
X_test, y_test = create_dataset(test, time_step)

In [ ]:
# RNN girişine uygun şekil verme
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1) #(örnek sayısı, zaman adımı, özellik sayısı)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [ ]:
# RNN modeli
model = Sequential()
model.add(SimpleRNN(50, return_sequences=True, input_shape=(time_step, 1)))
model.add(SimpleRNN(50))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# Eğitme
history = model.fit(X_train, y_train, epochs=20, batch_size=1, validation_data=(X_test, y_test))

# Tahmin
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)

# Tahminleri geri ölçeklendirme
train_predict = scaler.inverse_transform(train_predict)
test_predict = scaler.inverse_transform(test_predict)
y_train_actual = scaler.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
# Görselleştirme
plt.figure(figsize=(12,6))
plt.plot(data, label='Actual Data')
plt.plot(range(time_step, len(train_predict) + time_step), train_predict, label='Train Predict')
plt.plot(range(len(train_predict) + (time_step*2), len(train_predict) + (time_step*2) + len(test_predict)), test_predict, label='Test Predict')
plt.xlabel('Time')
plt.ylabel('Passengers')
plt.legend()
plt.show()

In [ ]:
# Değerlendirme
train_mae = mean_absolute_error(y_train_actual, train_predict)
test_mae = mean_absolute_error(y_test_actual, test_predict)
train_mse = mean_squared_error(y_train_actual, train_predict)
test_mse = mean_squared_error(y_test_actual, test_predict)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

print(f'Train MAE: {train_mae:.2f}')
print(f'Test MAE: {test_mae:.2f}')
print(f'Train MSE: {train_mse:.2f}')
print(f'Test MSE: {test_mse:.2f}')
print(f'Train RMSE: {train_rmse:.2f}')
print(f'Test RMSE: {test_rmse:.2f}')